## Transform Results Data
1. Read bronze results table
2. Keep only the columns required for analytics(Drop url Column)
3. Standardise column names using snake_case (constructorId -> constructor_id, driverId -> driver_id,raceName -> race_name, positionText -> finish_position_text)
4. Rename columns to make them more meaningful(date -> race_date, grid -> grid_position, laps -> completed_laps, number -> car_number, position -> finish_position)
5. Filter out rows where season, round, constructor_id or driver_id is null (business key validation)
6. Remove duplicate records
7. Transform values of columns race_name to Title Case
8. Write the transformed data to silver results table

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/03.silver-helpers

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.results"
silver_table = f"{catalog_name}.{silver_schema}.results"

#### Step 1 -  Read bronze results table, select only the required columns and standardise column names

In [0]:
from pyspark.sql import functions as F
results_df = (
    spark.read.table(bronze_table)
    .filter((F.col("batch_id") == v_batch_id))
    .drop(F.col("url"))
    .withColumnsRenamed({
        "constructorId":"constructor_id",
        "driverId" : "driver_id",
        "raceName" : "race_name",
        "positionText":"finish_position_text",
        "date" : "race_date",
        "grid" : "grid_position",
        "laps" : "completed_laps",
        "number" : "car_number",
        "position" : "finish_position"
    })
)

#### Step 5 & 6 Apply Data quality Checks
  - Filter out rows where season, round, constructor_id or driver_id is null (business key validation)
  - Remove duplicate records

In [0]:
results_valid_df = (
    results_df
    .filter(
        F.col("season").isNotNull() &
        F.col("round").isNotNull() &
        F.col("constructor_id").isNotNull() &
        F.col("driver_id").isNotNull()
        )
    .dropDuplicates(["season","round","constructor_id","driver_id"])
)

#### Step 7 - Transform values of columns nationality to Title Case

In [0]:
results_final_df = (
    results_valid_df
    .withColumn("race_name",F.initcap(F.col("race_date")))
)

#### Step 8 -  Write the transformed data to silver circuits table

In [0]:
merge_condition="tgt.season = src.season AND tgt.round = src.round AND tgt.constructor_id = src.constructor_id AND tgt.driver_id = src.driver_id"
columns_to_update=[
        "race_name",
        "race_date",
        "grid_position",
        "completed_laps",
        "car_number",
        "points",
        "finish_position",
        "finish_position_text",
        "status",
        "ingestion_timestamp",
        "source_file",
        "batch_id"
    ]

write_to_silver(results_final_df, silver_table, merge_condition, columns_to_update)


In [0]:
display(spark.table(silver_table))